In [1]:
# Imports and Configuration

import os
import re
import json
import time
import logging
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Tuple
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv
from groq import Groq
from ddgs import DDGS

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

MAX_RETRIES = 3
RETRY_DELAY = 2
DEFAULT_MODEL = "llama-3.3-70b-versatile"
DEFAULT_MAX_TOKENS = 2048
DEFAULT_TEMPERATURE = 0.3
MAX_RESULTS_PER_QUERY = 4
MAX_BODY_LENGTH = 300
HISTORY_PATH = "C:/educational files/advanced_agent/memory/search_history.json"

In [2]:
# Client Initialization

env_path = Path("C:/educational files/advanced_agent/.env")
load_dotenv(dotenv_path=env_path)

def init_client() -> Groq:
    api_key = os.getenv("GROQ_API_KEY")
    if not api_key:
        raise EnvironmentError("GROQ_API_KEY not found in .env")
    log.info("Groq client initialized successfully")
    return Groq(api_key=api_key)

client = init_client()

2026-06-02 18:46:24,386 [INFO] Groq client initialized successfully


In [3]:
# Agent Configuration

@dataclass
class AgentConfig:
    model: str = DEFAULT_MODEL
    max_tokens: int = DEFAULT_MAX_TOKENS
    temperature: float = DEFAULT_TEMPERATURE
    system_prompt: str = (
        "You are an advanced autonomous web search agent. "
        "You plan searches strategically, analyze multiple sources, rank them by credibility, "
        "and synthesize results into accurate, structured answers. "
        "When given search results, extract key facts and present them clearly with sources. "
        "Always distinguish between confirmed facts and uncertain claims. "
        "Be concise, precise, and professional."
    )
    session_token_count: int = field(default=0, repr=False)
    total_searches: int = field(default=0, repr=False)
    total_queries_planned: int = field(default=0, repr=False)

config = AgentConfig()
log.info(f"Agent configured — model: {config.model} | search agent ready")

2026-06-02 18:46:48,344 [INFO] Agent configured — model: llama-3.3-70b-versatile | search agent ready


In [4]:
# Query Planner

def plan_queries(user_input: str, cfg: AgentConfig) -> List[str]:
    prompt = (
        "You are a search query planner. "
        "Break the following question into 2-3 specific, focused search queries. "
        "Return ONLY a JSON array of strings, nothing else. No explanation, no markdown.\n"
        f"Question: {user_input}"
    )
    try:
        response = client.chat.completions.create(
            model=cfg.model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=256,
            temperature=0.2
        )
        raw = response.choices[0].message.content.strip()
        cfg.session_token_count += response.usage.total_tokens
        queries = json.loads(raw)
        if isinstance(queries, list) and all(isinstance(q, str) for q in queries):
            cfg.total_queries_planned += len(queries)
            log.info(f"Query planner generated {len(queries)} sub-queries")
            return queries
    except Exception as e:
        log.warning(f"Query planner failed: {e} — falling back to original input")
    return [user_input]

In [5]:
# Search Engine

def run_search(queries: List[str], cfg: AgentConfig) -> List[Dict]:
    seen_urls = set()
    all_results = []

    for query in queries:
        try:
            with DDGS() as ddgs:
                results = list(ddgs.text(query, max_results=MAX_RESULTS_PER_QUERY))
            cfg.total_searches += 1
            log.info(f"Search executed: '{query}' — {len(results)} results")

            for r in results:
                url = r.get("href", "")
                if url in seen_urls:
                    continue
                seen_urls.add(url)
                all_results.append({
                    "title": r.get("title", "").strip(),
                    "body": r.get("body", "")[:MAX_BODY_LENGTH].strip(),
                    "url": url,
                    "query": query
                })
        except Exception as e:
            log.warning(f"Search failed for query '{query}': {e}")

    log.info(f"Total unique results collected: {len(all_results)}")
    return all_results

In [6]:
# Result Extractor

def extract_results(raw_results: List[Dict]) -> List[Dict]:
    extracted = []
    for r in raw_results:
        title = r.get("title", "").strip()
        body = r.get("body", "").strip()
        url = r.get("url", "").strip()
        query = r.get("query", "").strip()

        if not title or not body:
            continue

        cleaned_body = re.sub(r'\s+', ' ', body)
        extracted.append({
            "title": title,
            "body": cleaned_body,
            "url": url,
            "query": query,
            "word_count": len(cleaned_body.split())
        })

    log.info(f"Extracted {len(extracted)} clean results from {len(raw_results)} raw results")
    return extracted

In [7]:
# Result Summarizer

def summarize_results(user_input: str, results: List[Dict], cfg: AgentConfig) -> str:
    if not results:
        return "No results found to summarize."

    results_text = "\n\n".join([
        f"Source {i+1}: {r['title']}\nURL: {r['url']}\nContent: {r['body']}"
        for i, r in enumerate(results)
    ])

    prompt = (
        f"User question: {user_input}\n\n"
        f"Search results:\n{results_text}\n\n"
        "Synthesize the above search results into a clear, accurate, structured answer. "
        "Mention sources where relevant. "
        "Distinguish confirmed facts from uncertain claims. "
        "Be concise and professional."
    )

    for attempt in range(MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=cfg.model,
                messages=[
                    {"role": "system", "content": cfg.system_prompt},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=cfg.max_tokens,
                temperature=cfg.temperature
            )
            summary = response.choices[0].message.content.strip()
            cfg.session_token_count += response.usage.total_tokens
            log.info(f"Summary generated — tokens session total: {cfg.session_token_count}")
            return summary
        except Exception as e:
            log.warning(f"Attempt {attempt + 1} failed: {e}")
            if attempt < MAX_RETRIES - 1:
                time.sleep(RETRY_DELAY)

    return "Failed to generate summary."

In [8]:
# Source Ranker

CREDIBLE_DOMAINS = {
    "wikipedia.org": 10, "bbc.com": 9, "reuters.com": 9,
    "nature.com": 9, "sciencedirect.com": 9, "arxiv.org": 8,
    "techcrunch.com": 7, "theverge.com": 7, "wired.com": 7,
    "github.com": 7, "stackoverflow.com": 6, "medium.com": 5
}

def rank_sources(results: List[Dict]) -> List[Dict]:
    def score(r: Dict) -> int:
        url = r.get("url", "").lower()
        word_count = r.get("word_count", 0)
        domain_score = 0
        for domain, pts in CREDIBLE_DOMAINS.items():
            if domain in url:
                domain_score = pts
                break
        length_score = min(word_count // 10, 5)
        return domain_score + length_score

    ranked = sorted(results, key=score, reverse=True)
    log.info(f"Sources ranked — top source: {ranked[0]['title'][:60] if ranked else 'none'}")
    return ranked

In [9]:
# Search History Manager

class SearchHistoryManager:
    def __init__(self, path: str = HISTORY_PATH):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.history: List[Dict] = self._load()
        log.info(f"Search history loaded — {len(self.history)} past searches")

    def _load(self) -> List[Dict]:
        if self.path.exists():
            try:
                return json.loads(self.path.read_text(encoding="utf-8"))
            except Exception:
                return []
        return []

    def save(self, query: str, sub_queries: List[str], results: List[Dict], summary: str) -> None:
        entry = {
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "query": query,
            "sub_queries": sub_queries,
            "result_count": len(results),
            "top_sources": [r["url"] for r in results[:3]],
            "summary": summary
        }
        self.history.append(entry)
        self.path.write_text(json.dumps(self.history, indent=2), encoding="utf-8")
        log.info(f"Search saved to history — total entries: {len(self.history)}")

    def get_last(self) -> Optional[Dict]:
        return self.history[-1] if self.history else None

    def get_all(self) -> List[Dict]:
        return self.history.copy()

    def clear(self) -> None:
        self.history.clear()
        self.path.write_text("[]", encoding="utf-8")
        log.info("Search history cleared")

    def summary(self) -> str:
        return f"Total searches stored: {len(self.history)}"

search_history = SearchHistoryManager()

2026-06-02 18:49:11,858 [INFO] Search history loaded — 0 past searches


In [10]:
# Follow-up Detector

def is_followup(user_input: str, history: SearchHistoryManager, cfg: AgentConfig) -> Tuple[bool, str]:
    last = history.get_last()
    if not last:
        return False, ""

    prompt = (
        "You are a follow-up detector. Determine if the new question is a follow-up to the previous search.\n"
        f"Previous search: {last['query']}\n"
        f"New question: {user_input}\n"
        "Reply ONLY with a JSON object: {\"is_followup\": true/false, \"reason\": \"one line explanation\"}"
    )

    try:
        response = client.chat.completions.create(
            model=cfg.model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=128,
            temperature=0.1
        )
        raw = response.choices[0].message.content.strip()
        cfg.session_token_count += response.usage.total_tokens
        parsed = json.loads(raw)
        result = parsed.get("is_followup", False)
        reason = parsed.get("reason", "")
        log.info(f"Follow-up detection: {result} — {reason}")
        return result, last["summary"] if result else ""
    except Exception as e:
        log.warning(f"Follow-up detector failed: {e}")
        return False, ""

In [11]:
# Web Search Agent

def search_agent(user_input: str, history: SearchHistoryManager, cfg: AgentConfig) -> str:
    log.info(f"Search agent started — query: {user_input[:80]}")

    # follow-up detection
    followup, prior_context = is_followup(user_input, history, cfg)
    if followup:
        log.info("Follow-up detected — injecting prior context")
        user_input_augmented = f"Previous context: {prior_context}\n\nFollow-up question: {user_input}"
    else:
        user_input_augmented = user_input

    # query planning
    sub_queries = plan_queries(user_input_augmented, cfg)
    log.info(f"Sub-queries planned: {sub_queries}")

    # search execution
    raw_results = run_search(sub_queries, cfg)

    # extraction
    extracted = extract_results(raw_results)

    # ranking
    ranked = rank_sources(extracted)

    # summarization
    summary = summarize_results(user_input, ranked, cfg)

    # save to history
    history.save(user_input, sub_queries, ranked, summary)

    return summary

In [12]:
# Interactive Chat Loop

print("Web Search Agent ready.")
print("Commands: 'exit' | 'history' to show past searches | 'last' to show last search | 'clear' to reset history | 'stats' for usage\n")

while True:
    user_input = input("You: ").strip()

    if not user_input:
        continue
    if user_input.lower() == "exit":
        print(f"Session ended. Tokens: {config.session_token_count} | Total searches: {config.total_searches} | Queries planned: {config.total_queries_planned}")
        break
    if user_input.lower() == "stats":
        print(f"Tokens: {config.session_token_count} | Searches: {config.total_searches} | Queries planned: {config.total_queries_planned} | {search_history.summary()}\n")
        continue
    if user_input.lower() == "history":
        all_history = search_history.get_all()
        if not all_history:
            print("No search history yet.\n")
        else:
            for i, entry in enumerate(all_history):
                print(f"{i+1}. [{entry['timestamp']}] {entry['query']}")
            print()
        continue
    if user_input.lower() == "last":
        last = search_history.get_last()
        if not last:
            print("No previous search found.\n")
        else:
            print(f"Query: {last['query']}")
            print(f"Sub-queries: {last['sub_queries']}")
            print(f"Sources: {last['top_sources']}")
            print(f"Summary: {last['summary']}\n")
        continue

    reply = search_agent(user_input, search_history, config)
    print(f"\nAgent: {reply}\n")

Web Search Agent ready.
Commands: 'exit' | 'history' to show past searches | 'last' to show last search | 'clear' to reset history | 'stats' for usage



You:  what is quantum computing


2026-06-02 18:50:50,972 [INFO] Search agent started — query: what is quantum computing
2026-06-02 18:50:52,145 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-02 18:50:52,195 [INFO] Query planner generated 3 sub-queries
2026-06-02 18:50:52,196 [INFO] Sub-queries planned: ['what is quantum computing', 'quantum computing definition', 'how does quantum computing work']
2026-06-02 18:50:53,293 [INFO] response: https://en.wikipedia.org/w/api.php?action=opensearch&profile=fuzzy&limit=1&search=what%20is%20quantum%20computing 200
2026-06-02 18:50:53,926 [INFO] response: https://grokipedia.com/api/typeahead?query=what+is+quantum+computing&limit=1 200
2026-06-02 18:50:56,248 [INFO] response: https://www.mojeek.com/search?q=what+is+quantum+computing 200
2026-06-02 18:50:56,570 [INFO] Search executed: 'what is quantum computing' — 4 results
2026-06-02 18:50:57,145 [INFO] response: https://grokipedia.com/api/typeahead?query=quantum+computing+defin


Agent: **Definition of Quantum Computing:**
Quantum computing is a branch of physics that harnesses the principles of quantum mechanics to perform calculations and operations on data. It is based on the strange and often counterintuitive laws that govern the universe at its smallest scales and coldest temperatures (Source: NIST).

**How Quantum Computing Works:**
Quantum computers work by using quantum bits or qubits, which are the fundamental units of quantum information. Qubits are unique because they can exist in multiple states simultaneously, allowing for parallel processing and potentially solving complex problems more efficiently than classical computers (Source: IBM).

**Confirmed Facts:**

1. Quantum computing is a field of research that combines physics, computer science, and engineering to develop new types of computers (Source: Wikipedia).
2. Quantum computers have the potential to solve certain problems much faster than classical computers, making them useful for applicat

You:  how does it compare to classical computing


...

1. [2026-06-02 18:51:03] what is quantum computing
2. [2026-06-02 18:51:27] how does it compare to classical computing



You:  last


Query: how does it compare to classical computing
Sub-queries: ['quantum computing vs classical computing', 'comparison of quantum and classical computing principles', 'classical computing limitations and quantum computing advantages']
Sources: ['https://en.wikipedia.org/wiki/Quantum_computing', 'https://www.techtarget.com/searchdatacenter/tip/Classical-vs-quantum-computing-What-are-the-differences', 'https://www.berkeleynucleonics.com/august-23-2024-quantum-computing-vs-classical-computing/']
Summary: **Comparison of Quantum Computing and Classical Computing**

Quantum computing and classical computing differ significantly in their principles, operations, and capabilities. The key differences between the two are:

1. **Basic Principle**: Classical computers process information using bits that exist as definite 0s or 1s, whereas quantum computers use qubits that can exist in superposition (0, 1, or both simultaneously) and leverage entanglement (Source 6, Source 8).
2. **Compute Power*

You:  exit


Session ended. Tokens: 3361 | Total searches: 6 | Queries planned: 6
